In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.transforms import trainval_transforms, revert_normalization, revert_standardization
from src.dataset import ImageDataset
import torch

annot_path = Path("../data/preprocessed/trainval/annotations.csv")
img_dir = Path("../data/preprocessed/trainval/images")

# test dataset without transforms
dataset = ImageDataset(annot_path, img_dir, transform=trainval_transforms)

In [3]:
from src.model import Model
from torch.utils.data import DataLoader

trainval_dl = DataLoader(dataset, 8, True)
X_batch, y_batch = next(iter(trainval_dl))
model = Model()

model.eval()
with torch.no_grad():
    preds_batch = model(X_batch)

In [4]:
from src.postprocessing import postprocess_preds

postprocessed_preds = postprocess_preds(preds_batch[0])

In [5]:
postprocessed_preds

{'horse': [(tensor(7.8940e-05),
   tensor(31.3888),
   tensor(1.4610),
   tensor(31.4540),
   tensor(-1.1607))],
 'bird': [(tensor(0.0002),
   tensor(95.7002),
   tensor(192.7498),
   tensor(95.1025),
   tensor(190.1396)),
  (tensor(0.0002),
   tensor(95.1593),
   tensor(0.0375),
   tensor(98.4509),
   tensor(-0.1213)),
  (tensor(6.7019e-05),
   tensor(34.2200),
   tensor(161.8143),
   tensor(30.9989),
   tensor(157.6411))],
 'bottle': [(tensor(0.0002),
   tensor(98.0960),
   tensor(96.7660),
   tensor(94.9989),
   tensor(95.6585)),
  (tensor(0.0002),
   tensor(158.7165),
   tensor(160.2642),
   tensor(161.7210),
   tensor(160.1100)),
  (tensor(3.1087e-05),
   tensor(126.3045),
   tensor(-0.7764),
   tensor(129.3816),
   tensor(1.4436))],
 'boat': [(tensor(0.0004),
   tensor(-0.8837),
   tensor(126.9823),
   tensor(1.3879),
   tensor(129.3334)),
  (tensor(0.0002),
   tensor(-0.3230),
   tensor(191.0802),
   tensor(0.4440),
   tensor(192.4218)),
  (tensor(0.0001),
   tensor(160.7005),
 

In [6]:
from src.utilities import objects_in_target

ground_truth_objects = objects_in_target(y_batch[0])

In [7]:
ground_truth_objects

{'tvmonitor': [(tensor(156.5000),
   tensor(86.5000),
   tensor(177.5000),
   tensor(101.5000)),
  (tensor(94.), tensor(82.), tensor(130.), tensor(122.))],
 'bottle': [(tensor(195.5000), tensor(88.), tensor(198.5000), tensor(102.)),
  (tensor(200.), tensor(119.), tensor(204.), tensor(133.))],
 'pottedplant': [(tensor(11.5000),
   tensor(93.5000),
   tensor(18.5000),
   tensor(134.5000)),
  (tensor(31.), tensor(109.5000), tensor(47.), tensor(132.5000)),
  (tensor(65.), tensor(97.), tensor(73.), tensor(129.)),
  (tensor(135.5000), tensor(38.5000), tensor(160.5000), tensor(159.5000)),
  (tensor(40.5000), tensor(121.5000), tensor(57.5000), tensor(158.5000)),
  (tensor(144.5000), tensor(115.5000), tensor(159.5000), tensor(158.5000))],
 'chair': [(tensor(160.), tensor(92.), tensor(188.), tensor(146.)),
  (tensor(170.5000), tensor(133.5000), tensor(223.5000), tensor(216.5000)),
  (tensor(154.), tensor(186.5000), tensor(224.), tensor(223.5000))],
 'sofa': [(tensor(0.), tensor(154.), tensor(34.

In [8]:
tp_fp_by_class = {
    "aeroplane": [],
    "bicycle": [],
    "bird": [],
    "boat": [],
    "bottle": [],
    "bus": [],
    "car": [],
    "cat": [],
    "chair": [],
    "cow": [],
    "diningtable": [],
    "dog": [],
    "horse": [],
    "motorbike": [],
    "person": [],
    "pottedplant": [],
    "sheep": [],
    "sofa": [],
    "train": [],
    "tvmonitor": [],
}

In [9]:
from src.evaluation import find_tp_and_fp

find_tp_and_fp(postprocessed_preds, ground_truth_objects, 
               tp_fp_by_class)

{'aeroplane': [], 'bicycle': [], 'bird': [(tensor(0.0002), False), (tensor(0.0002), False), (tensor(6.7019e-05), False)], 'boat': [], 'bottle': [], 'bus': [], 'car': [], 'cat': [], 'chair': [], 'cow': [], 'diningtable': [], 'dog': [], 'horse': [(tensor(7.8940e-05), False)], 'motorbike': [], 'person': [], 'pottedplant': [], 'sheep': [], 'sofa': [], 'train': [], 'tvmonitor': []} 

{'tvmonitor': [(tensor(156.5000), tensor(86.5000), tensor(177.5000), tensor(101.5000)), (tensor(94.), tensor(82.), tensor(130.), tensor(122.))], 'bottle': [(tensor(195.5000), tensor(88.), tensor(198.5000), tensor(102.)), (tensor(200.), tensor(119.), tensor(204.), tensor(133.))], 'pottedplant': [(tensor(11.5000), tensor(93.5000), tensor(18.5000), tensor(134.5000)), (tensor(31.), tensor(109.5000), tensor(47.), tensor(132.5000)), (tensor(65.), tensor(97.), tensor(73.), tensor(129.)), (tensor(135.5000), tensor(38.5000), tensor(160.5000), tensor(159.5000)), (tensor(40.5000), tensor(121.5000), tensor(57.5000), tensor

In [10]:
tp_fp_by_class

{'aeroplane': [],
 'bicycle': [],
 'bird': [(tensor(0.0002), False),
  (tensor(0.0002), False),
  (tensor(6.7019e-05), False)],
 'boat': [(tensor(0.0004), False),
  (tensor(0.0002), False),
  (tensor(0.0001), False),
  (tensor(5.1963e-05), False)],
 'bottle': [(tensor(0.0002), True),
  (tensor(0.0002), False),
  (tensor(3.1087e-05), False)],
 'bus': [(tensor(0.0001), False)],
 'car': [(tensor(0.0003), False), (tensor(7.6553e-06), False)],
 'cat': [],
 'chair': [],
 'cow': [(tensor(0.0004), False),
  (tensor(0.0001), False),
  (tensor(9.9017e-05), False),
  (tensor(5.4196e-06), False),
  (tensor(1.0955e-06), False)],
 'diningtable': [],
 'dog': [],
 'horse': [(tensor(7.8940e-05), False)],
 'motorbike': [(tensor(0.0004), False)],
 'person': [],
 'pottedplant': [(tensor(0.0002), True), (tensor(0.0002), False)],
 'sheep': [(tensor(0.0003), False), (tensor(0.0002), False)],
 'sofa': [],
 'train': [],
 'tvmonitor': []}